# 📊 Ablation Study — HotpotQA Benchmark

**Autonomous Researcher — Experiment Analysis Report**

> Benchmark: [HotpotQA](https://hotpotqa.github.io/) — Multi-hop factual question answering  
> Questions evaluated: **14**  
> Data source: `notebooks/logs/experiments_hotpotqa/`

---

## 🔬 Experiment Overview

| ID | System | Description |
|---|---|---|
| EXP1 | **Vanilla LLM** | Direct generation, no search, no verification |
| EXP2 | **RAG Baseline** | Retrieval-Augmented Generation only |
| EXP3 | **Agent Base** | Multi-agent with planning, web search & scraping |
| EXP4 | **Agent + FT Reviewer** | Agent + LoRA fine-tuned reviewer |
| EXP5 | **Agent + FT Reranker** | Agent + fine-tuned cross-encoder reranker |
| EXP6 | **Agent Full (System-2)** | Agent + both FT Reviewer & FT Reranker |

## 📐 Metrics

| Metric | Description | Range |
|---|---|---|
| `mean_f1_score` | Token-overlap F1 vs ground truth answer | 0–1 |
| `mean_citation_precision` | Fraction of citations grounded in retrieved evidence | 0–1 |
| `mean_judge_comprehensiveness` | LLM-as-judge: completeness of the report | 1–5 |
| `mean_judge_depth` | LLM-as-judge: depth of analysis | 1–5 |
| `mean_overall_judge` | LLM-as-judge: holistic quality | 1–5 |
| `mean_step_count` | Average number of agent reasoning steps | — |
| `success_rate` | Fraction of questions answered without crash | 0–1 |

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from math import pi

warnings.filterwarnings('ignore')

# ── Aesthetics ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.6,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})

PALETTE = ['#58a6ff', '#3fb950', '#d2a8ff', '#ffa657', '#ff7b72', '#56d364']
EXP_LABELS = [
    'EXP1\nVanilla LLM',
    'EXP2\nRAG Baseline',
    'EXP3\nAgent Base',
    'EXP4\nAgent+\nFT-Reviewer',
    'EXP5\nAgent+\nFT-Reranker',
    'EXP6\nAgent Full',
]
EXP_SHORT = ['EXP1', 'EXP2', 'EXP3', 'EXP4', 'EXP5', 'EXP6']

print('Libraries loaded ✓')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'logs', 'experiments_hotpotqa')
# Fallback: try relative path if running from notebooks/
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'logs/experiments_hotpotqa'
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'logs', 'experiments_hotpotqa')

csv_path = os.path.join(DATA_DIR, 'experiments_comparison.csv')
df = pd.read_csv(csv_path, index_col=0)
df.index = EXP_SHORT

print('Loaded data from:', csv_path)
df.round(4)

---
## 1️⃣ Summary Statistics

In [ ]:
METRICS = [
    'mean_f1_score', 'mean_citation_precision',
    'mean_judge_comprehensiveness', 'mean_judge_depth',
    'mean_overall_judge', 'mean_step_count', 'success_rate'
]
METRIC_DISPLAY = [
    'F1 Score', 'Citation Precision',
    'Comprehensiveness', 'Depth',
    'Overall Judge', 'Step Count', 'Success Rate'
]

summary = df[METRICS].copy()

# Highlight best values
def highlight_best(col):
    if col.name == 'mean_step_count':
        return ['background-color: #1f3d2a' if v == col.min() else '' for v in col]
    return ['background-color: #1f3d2a' if v == col.max() else '' for v in col]

try:
    import jinja2
    has_jinja2 = True
except ImportError:
    has_jinja2 = False

if has_jinja2:
    display(summary.style
        .apply(highlight_best)
        .format({
            'mean_f1_score': '{:.4f}',
            'mean_citation_precision': '{:.4f}',
            'mean_judge_comprehensiveness': '{:.3f}',
            'mean_judge_depth': '{:.3f}',
            'mean_overall_judge': '{:.3f}',
            'mean_step_count': '{:.1f}',
            'success_rate': '{:.1%}',
        })
        .set_caption('HotpotQA Benchmark Results (green = best per column, step_count = lower is better)')
    )
else:
    # Fallback if jinja2 is not installed
    formatted_summary = summary.copy()
    for col in formatted_summary.columns:
        if col in ['mean_f1_score', 'mean_citation_precision']:
            formatted_summary[col] = formatted_summary[col].map('{:.4f}'.format)
        elif col in ['mean_judge_comprehensiveness', 'mean_judge_depth', 'mean_overall_judge']:
            formatted_summary[col] = formatted_summary[col].map('{:.3f}'.format)
        elif col == 'mean_step_count':
            formatted_summary[col] = formatted_summary[col].map('{:.1f}'.format)
        elif col == 'success_rate':
            formatted_summary[col] = formatted_summary[col].map('{:.1%}'.format)
    print("Note: Install 'jinja2' to enable color highlighting in the table below.")
    display(formatted_summary)


---
## 2️⃣ Bar Charts — All Metrics

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('HotpotQA — All Metrics per Experiment', fontsize=18, fontweight='bold', color='#e6edf3', y=1.01)
axes_flat = axes.flatten()

x = np.arange(len(EXP_SHORT))

for i, (metric, label) in enumerate(zip(METRICS, METRIC_DISPLAY)):
    ax = axes_flat[i]
    values = df[metric].values
    
    bars = ax.bar(x, values, color=PALETTE, width=0.6, edgecolor='#0d1117', linewidth=0.8)
    
    # Value labels on top of bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(values),
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, color='#e6edf3', fontweight='bold')
    
    # Highlight best bar
    best_idx = int(np.argmin(values) if metric == 'mean_step_count' else np.argmax(values))
    bars[best_idx].set_edgecolor('#f0e130')
    bars[best_idx].set_linewidth(2.5)
    
    ax.set_title(label, pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(EXP_SHORT, fontsize=8.5)
    ax.grid(axis='y', alpha=0.4)
    ax.set_ylim(0, max(values) * 1.2 + 0.05)

# Hide unused subplot
axes_flat[-1].set_visible(False)
axes_flat[-2].set_visible(False)

legend_patches = [mpatches.Patch(color=PALETTE[i], label=f'{EXP_SHORT[i]}: {EXP_LABELS[i].replace(chr(10), " ")}') for i in range(6)]
fig.legend(handles=legend_patches, loc='lower right', ncol=3, bbox_to_anchor=(0.98, 0.03), fontsize=9)

plt.tight_layout()
plt.savefig('hotpotqa_all_metrics_bar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Figure saved: hotpotqa_all_metrics_bar.png')

---
## 3️⃣ 🕸️ Radar Chart (Spider Chart) — All Experiments

In [ ]:
# Normalize metrics to [0, 1] for radar chart
# For step_count: invert so more steps = lower score (efficiency matters)
radar_metrics = [
    'mean_f1_score', 'mean_citation_precision',
    'mean_judge_comprehensiveness', 'mean_judge_depth',
    'mean_overall_judge', 'mean_step_count'
]
radar_labels = [
    'F1 Score', 'Citation\nPrecision',
    'Comprehensiveness', 'Depth',
    'Overall Judge', 'Efficiency\n(fewer steps)'
]

radar_data = df[radar_metrics].copy()

# Normalize each metric
normalized = radar_data.copy()
for col in radar_metrics:
    col_min, col_max = radar_data[col].min(), radar_data[col].max()
    if col_max - col_min > 0:
        if col == 'mean_step_count':
            # Invert: fewer steps = better
            normalized[col] = 1 - (radar_data[col] - col_min) / (col_max - col_min)
        else:
            normalized[col] = (radar_data[col] - col_min) / (col_max - col_min)
    else:
        normalized[col] = 1.0

N = len(radar_metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(1, 1, figsize=(10, 10), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

# Draw grid circles
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='#8b949e')
ax.yaxis.grid(True, color='#21262d', linestyle='--', alpha=0.8)
ax.xaxis.grid(True, color='#21262d', linestyle='--', alpha=0.8)
ax.spines['polar'].set_color('#30363d')

# Set axis labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11, color='#c9d1d9', fontweight='bold')

# Plot each experiment
for i, (exp_id, color) in enumerate(zip(EXP_SHORT, PALETTE)):
    values_norm = normalized.loc[exp_id, radar_metrics].tolist()
    values_norm += values_norm[:1]
    
    ax.plot(angles, values_norm, 'o-', linewidth=2, color=color, label=exp_id, markersize=5)
    ax.fill(angles, values_norm, alpha=0.08, color=color)

ax.set_title('HotpotQA — Multi-Metric Radar Comparison\n(all axes normalized to [0,1], efficiency = 1 − normalized_steps)',
             fontsize=13, fontweight='bold', color='#e6edf3', pad=25)

legend = ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
for text in legend.get_texts():
    text.set_color('#c9d1d9')

plt.tight_layout()
plt.savefig('hotpotqa_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Radar chart saved: hotpotqa_radar.png')

---
## 4️⃣ Heatmap — Normalized Metric Matrix

In [ ]:
# Build normalized matrix (same as radar but all 7 metrics)
norm_all = df[METRICS].copy().astype(float)
for col in METRICS:
    col_min, col_max = norm_all[col].min(), norm_all[col].max()
    if col_max - col_min > 0:
        if col == 'mean_step_count':
            norm_all[col] = 1 - (norm_all[col] - col_min) / (col_max - col_min)
        else:
            norm_all[col] = (norm_all[col] - col_min) / (col_max - col_min)
    else:
        norm_all[col] = 1.0

norm_all.columns = METRIC_DISPLAY

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    norm_all,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5, linecolor='#0d1117',
    ax=ax,
    cbar_kws={'label': 'Normalized Score (higher = better)', 'shrink': 0.8}
)

ax.set_title('HotpotQA — Normalized Performance Heatmap\n(Step Count inverted: higher = more efficient)',
             fontsize=14, fontweight='bold', color='#e6edf3', pad=15)
ax.set_yticklabels(EXP_SHORT, rotation=0, fontsize=11)
ax.set_xticklabels(METRIC_DISPLAY, rotation=30, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('hotpotqa_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Heatmap saved: hotpotqa_heatmap.png')

---
## 5️⃣ Key Metric Deep-Dives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('HotpotQA — F1 Score vs Citation Precision Trade-off', fontsize=15, fontweight='bold', color='#e6edf3')

# ── Left: grouped bar F1 vs Citation Precision ──
ax = axes[0]
x = np.arange(len(EXP_SHORT))
w = 0.35

b1 = ax.bar(x - w/2, df['mean_f1_score'], w, label='F1 Score', color='#58a6ff', edgecolor='#0d1117')
b2 = ax.bar(x + w/2, df['mean_citation_precision'], w, label='Citation Precision', color='#3fb950', edgecolor='#0d1117')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8, color='#e6edf3')

ax.set_xticks(x)
ax.set_xticklabels(EXP_SHORT)
ax.set_ylabel('Score')
ax.set_title('F1 vs Citation Precision')
ax.legend()
ax.grid(axis='y', alpha=0.4)
ax.set_ylim(0, 1.25)

# Add annotation about the trade-off
ax.annotate('Vanilla/RAG: high F1\nbut zero citations',
            xy=(0.5, 0.115), xytext=(1.5, 0.6),
            arrowprops=dict(arrowstyle='->', color='#ffa657'),
            color='#ffa657', fontsize=9)
ax.annotate('Agent systems:\nhigh citation precision',
            xy=(2.5, 0.917), xytext=(3.5, 0.6),
            arrowprops=dict(arrowstyle='->', color='#56d364'),
            color='#56d364', fontsize=9)

# ── Right: judge scores comparison ──
ax = axes[1]
judge_metrics = ['mean_judge_comprehensiveness', 'mean_judge_depth', 'mean_overall_judge']
judge_labels = ['Comprehensiveness', 'Depth', 'Overall']
judge_colors = ['#d2a8ff', '#ffa657', '#ff7b72']

w = 0.22
offsets = [-w, 0, w]
for j, (metric, label, color) in enumerate(zip(judge_metrics, judge_labels, judge_colors)):
    bars = ax.bar(x + offsets[j], df[metric], w, label=label, color=color, edgecolor='#0d1117')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.03, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7.5, color='#e6edf3')

ax.set_xticks(x)
ax.set_xticklabels(EXP_SHORT)
ax.set_ylabel('Judge Score (1–5)')
ax.set_title('LLM-Judge Scores (Comprehensiveness, Depth, Overall)')
ax.set_ylim(0, 6.5)
ax.axhline(y=3.0, color='#58a6ff', linestyle='--', alpha=0.5, linewidth=1, label='Mid-point (3.0)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('hotpotqa_key_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Step count & efficiency ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0d1117')

steps = df['mean_step_count'].values
overall = df['mean_overall_judge'].values

sc = ax.scatter(steps, overall, c=PALETTE, s=280, zorder=5, edgecolors='white', linewidths=1.5)

for i, exp in enumerate(EXP_SHORT):
    ax.annotate(exp, (steps[i], overall[i]),
                textcoords='offset points', xytext=(10, 4),
                fontsize=10, color=PALETTE[i], fontweight='bold')

ax.set_xlabel('Mean Step Count (complexity)', fontsize=12)
ax.set_ylabel('Mean Overall Judge Score', fontsize=12)
ax.set_title('HotpotQA — Quality vs Complexity Trade-off', fontsize=14, fontweight='bold', color='#e6edf3')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#161b22')

# Ideal quadrant annotation
ax.axvline(x=np.mean(steps), color='#58a6ff', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(y=np.mean(overall), color='#58a6ff', linestyle='--', alpha=0.4, linewidth=1)
ax.text(2.5, np.mean(overall) + 0.05, '← fewer steps, higher quality (ideal →)', 
        color='#58a6ff', fontsize=9, alpha=0.7)

plt.tight_layout()
plt.savefig('hotpotqa_quality_vs_complexity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 6️⃣ 📝 Detailed Analysis

In [ ]:
print('=' * 70)
print('HOTPOTQA BENCHMARK — DETAILED ANALYSIS')
print('=' * 70)

for i, (exp_id, row) in enumerate(df.iterrows()):
    label_map = {
        'EXP1': 'Vanilla LLM (System-1)',
        'EXP2': 'RAG Baseline',
        'EXP3': 'Agent Base (Multi-agent, no fine-tuning)',
        'EXP4': 'Agent + Fine-Tuned Reviewer (LoRA)',
        'EXP5': 'Agent + Fine-Tuned Reranker (CrossEncoder)',
        'EXP6': 'Agent Full — System-2 (EXP3 + EXP4 + EXP5)',
    }
    print(f'\n── {exp_id}: {label_map.get(exp_id, exp_id)} ──')
    print(f'  F1 Score           : {row["mean_f1_score"]:.4f}')
    print(f'  Citation Precision : {row["mean_citation_precision"]:.4f}')
    print(f'  Comprehensiveness  : {row["mean_judge_comprehensiveness"]:.3f} / 5')
    print(f'  Depth              : {row["mean_judge_depth"]:.3f} / 5')
    print(f'  Overall Judge      : {row["mean_overall_judge"]:.3f} / 5')
    print(f'  Mean Steps         : {row["mean_step_count"]:.1f}')
    print(f'  Success Rate       : {row["success_rate"]:.1%}')

---
## 7️⃣ 🔍 Findings & Conclusions

### Key Observations

#### 1. F1 Score Paradox — Vanilla Wins at Token Matching
- **EXP1 (Vanilla LLM)** achieves the highest F1 score (`0.1152`) despite using no retrieval. This is because F1 token-overlap rewards concise, direct answers — which the Vanilla LLM produces by simply regurgitating factual patterns from pretraining data.
- Agent-based systems generate longer, more comprehensive reports which *dilute* F1 overlap with short ground-truth answers.
- **Conclusion:** F1 is a poor proxy for research quality in long-form generation tasks.

#### 2. Citation Precision — Agents vs Non-Agents
- **EXP1 & EXP2**: Citation precision = `0.0`. Neither system grounds citations in retrieved evidence — they either hallucinate citations or skip them.
- **EXP3, EXP4, EXP5**: Citation precision jumps to `0.917–0.929`. The multi-agent retrieval pipeline produces verifiable citations.
- **EXP6 (Full)**: Achieves perfect citation precision (`1.0`) — every citation is traceable to retrieved evidence.

#### 3. LLM-Judge Scores — EXP1 Scores Surprisingly High
- EXP1 scores highest on comprehensiveness (`4.857`) and depth (`4.286`). This is a **judge calibration artifact**: the LLM judge rewards fluent, confident prose (which Vanilla LLM produces) over factually-grounded but more cautious agentic reports.
- EXP4 (FT Reviewer) scores very low (`1.857` comprehensiveness) — the fine-tuned reviewer appears overly strict, frequently interrupting the pipeline before a complete report is generated.

#### 4. EXP5 (FT Reranker) is the Sweet Spot
- EXP5 achieves the best **balance**: high citation precision (`0.929`), competitive judge scores (overall `3.679`), and reasonable step count (`8.6`).
- The cross-encoder reranker improves evidence selection quality without the instability introduced by the LoRA reviewer.

#### 5. EXP6 (Full System) Does Not Beat EXP5
- Combining both fine-tuned components does **not** produce additive gains. The interaction between the FT reviewer (which tends to trigger more reflection loops) and the reranker increases steps (`7.2`) while reducing judge scores relative to EXP5.
- Hypothesis: the LoRA reviewer's feedback degrades the writer's final output quality under the multi-hop reasoning pressure of HotpotQA.

### 📌 Summary Table

| | Best | Worst | Winner |
|---|---|---|---|
| **F1 Score** | EXP1 (0.115) | EXP4 (0.033) | EXP1 |
| **Citation Precision** | EXP6 (1.000) | EXP1 (0.000) | EXP6 |
| **Comprehensiveness** | EXP1 (4.857) | EXP4 (1.857) | EXP1 |
| **Depth** | EXP1 (4.286) | EXP4 (1.821) | EXP1 |
| **Overall Judge** | EXP1 (4.643) | EXP4 (1.814) | EXP1 |
| **Efficiency (steps↓)** | EXP1 & EXP2 (2.0) | EXP5 (8.6) | EXP1/EXP2 |
| **Grounded Quality** | EXP5/EXP6 | EXP1/EXP2 | **EXP5** |

> **Recommended System**: EXP5 (Agent + Fine-Tuned Reranker) for production use — best balance of citation grounding, report quality, and efficiency.

In [ ]:
# ── Final combined summary figure ───────────────────────────────
fig = plt.figure(figsize=(20, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('HotpotQA Benchmark — Quick Reference Summary', fontsize=16, fontweight='bold', color='#e6edf3', y=1.02)

gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Panel 1: F1 + Citation
ax1 = fig.add_subplot(gs[0])
x = np.arange(len(EXP_SHORT))
w = 0.35
ax1.bar(x - w/2, df['mean_f1_score'], w, color='#58a6ff', label='F1', edgecolor='#0d1117')
ax1.bar(x + w/2, df['mean_citation_precision'], w, color='#3fb950', label='Citation P.', edgecolor='#0d1117')
ax1.set_xticks(x); ax1.set_xticklabels(EXP_SHORT, fontsize=9)
ax1.set_title('F1 & Citation Precision'); ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.3); ax1.grid(axis='y', alpha=0.3)

# Panel 2: Overall judge
ax2 = fig.add_subplot(gs[1])
colors = [PALETTE[i] for i in range(6)]
ax2.bar(x, df['mean_overall_judge'], color=colors, edgecolor='#0d1117')
ax2.axhline(3.0, color='white', linestyle='--', alpha=0.4, linewidth=1)
for xi, val in zip(x, df['mean_overall_judge']):
    ax2.text(xi, val + 0.05, f'{val:.2f}', ha='center', fontsize=9, color='#e6edf3')
ax2.set_xticks(x); ax2.set_xticklabels(EXP_SHORT, fontsize=9)
ax2.set_title('Overall Judge Score (1–5)'); ax2.set_ylim(0, 6)
ax2.grid(axis='y', alpha=0.3)

# Panel 3: Step count
ax3 = fig.add_subplot(gs[2])
ax3.bar(x, df['mean_step_count'], color=colors, edgecolor='#0d1117')
for xi, val in zip(x, df['mean_step_count']):
    ax3.text(xi, val + 0.1, f'{val:.1f}', ha='center', fontsize=9, color='#e6edf3')
ax3.set_xticks(x); ax3.set_xticklabels(EXP_SHORT, fontsize=9)
ax3.set_title('Mean Step Count (↓ more efficient)')
ax3.grid(axis='y', alpha=0.3)

plt.savefig('hotpotqa_summary.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Summary figure saved: hotpotqa_summary.png')
print('\n✅ HotpotQA analysis complete.')